# Analisis Komentar YouTube tentang AI

Notebook ini merangkum hasil pipeline text mining yang sudah dijalankan. Isinya membaca dataset dan file ringkasan di folder `dataset/` dan `results/`, lalu menampilkan angka utama serta visualisasi.

**Cara menjalankan:** buka notebook dari root proyek atau folder `notebook/`, lalu jalankan semua sel. Notebook tidak melakukan scraping ulang dan tidak menghitung stemming ulang; semua hasil dibaca dari file yang dihasilkan skrip pipeline.

Urutan pipeline lengkap ada di `README.md`.

In [ ]:
import sys
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.image import imread

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA = PROJECT_ROOT / "dataset"
RESULTS = PROJECT_ROOT / "results"
WORKFLOW = PROJECT_ROOT / "workflow"

print("Root proyek:", PROJECT_ROOT)
for name in ["raw_comments.csv", "cleaned_comments.csv", "preprocessed_comments.csv", "noise_comments.csv"]:
    path = DATA / name
    print(f"{name:30s} {'OK' if path.exists() else 'TIDAK ADA'}")

## 1. Dataset

Empat file utama yang dihasilkan pipeline:

| File | Isi |
|---|---|
| `raw_comments.csv` | Komentar mentah hasil scraping |
| `cleaned_comments.csv` | Komentar setelah cleaning |
| `preprocessed_comments.csv` | Komentar setelah normalisasi, stopword removal, dan stemming |
| `noise_comments.csv` | Komentar yang berlabel noise struktural |

In [ ]:
raw = pd.read_csv(DATA / "raw_comments.csv", keep_default_na=False)
cleaned = pd.read_csv(DATA / "cleaned_comments.csv", keep_default_na=False)
pre = pd.read_csv(DATA / "preprocessed_comments.csv", keep_default_na=False)
noise = pd.read_csv(DATA / "noise_comments.csv", keep_default_na=False)

print(f"Raw          : {len(raw):,} baris | {raw['comment_id'].nunique():,} comment_id unik | {raw['video_id'].nunique()} video")
print(f"Cleaned      : {len(cleaned):,} baris | {len(cleaned.columns)} kolom")
print(f"Preprocessed : {len(pre):,} baris | {len(pre.columns)} kolom")
print(f"Noise        : {len(noise):,} baris | {noise['comment_id'].nunique():,} comment_id unik")
raw.head(3)

## 2. Exploratory Data Analysis

Ringkasan EDA mencakup jumlah baris, kelengkapan kolom, distribusi panjang komentar, distribusi per video, dan indikasi noise. File lengkap: `results/eda_summary.txt`.

In [ ]:
def show_summary(filename):
    path = RESULTS / filename
    print(path.read_text(encoding="utf-8"))

show_summary("eda_summary.txt")

## 3. Cleaning

Cleaning melakukan case folding, menghapus URL/mention/hashtag/emoji/angka/tanda baca, merapikan spasi, lalu membuang komentar kosong dan duplikat. Kata tidak baku sengaja dipertahankan untuk dinormalisasi pada tahap berikutnya.

In [ ]:
show_summary("cleaning_summary.txt")

## 4. Preprocessing

Preprocessing terdiri dari tokenization, normalisasi dengan kamus 126 entri, stopword removal dengan 210 kata, dan stemming Sastrawi dengan 89 pengecualian. Istilah penting tentang AI dilindungi agar tidak dibuang atau dirusak.

In [ ]:
show_summary("preprocessing_summary.txt")

## 5. Noise Analysis

Label noise bersifat struktural: kosong, duplikat, terlalu pendek, karakter berulang, kata berulang, simbol berlebihan, dan spam. Kata informal seperti `gak` atau `bgt` tidak dianggap noise.

In [ ]:
show_summary("noise_summary.txt")

## 6. Frekuensi Kata

Tabel frekuensi dihitung untuk empat tahap: raw, cleaned, preprocessed, dan noise. Berikut 20 kata teratas hasil preprocessing beserta persentasenya terhadap total token akhir (33.264).

In [ ]:
top20 = pd.read_csv(RESULTS / "top_20_words.csv")
top20

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(imread(RESULTS / "top_20_words.png"))
ax.set_title("Bar chart Top 20 kata (hasil preprocessing)", fontsize=14, pad=12)
ax.axis("off")
plt.tight_layout()
plt.show()

## 7. WordCloud

Empat WordCloud dibuat dari tahap yang berbeda. Perhatikan bagaimana kata fungsi mendominasi raw/cleaned, lalu hilang setelah preprocessing.

In [ ]:
wordcloud_files = [
    "wordcloud_raw.png",
    "wordcloud_cleaned.png",
    "wordcloud_preprocessed.png",
    "wordcloud_noise.png",
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, name in zip(axes.ravel(), wordcloud_files):
    ax.imshow(imread(RESULTS / name))
    ax.set_title(name.replace("wordcloud_", "").replace(".png", ""), fontsize=13)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Workflow

Diagram workflow dibuat otomatis dari dataset dan file konfigurasi, sehingga angka pada diagram mengikuti hasil terbaru.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 9))
ax.imshow(imread(WORKFLOW / "workflow.png"))
ax.set_title("Workflow pipeline analisis komentar YouTube tentang AI", fontsize=14, pad=12)
ax.axis("off")
plt.tight_layout()
plt.show()

## 9. Ringkasan

- 3.000 komentar mentah dari 5 video; 3.000 `comment_id` unik.
- Cleaning menyisakan 2.935 komentar (97,8%) setelah membuang 22 baris kosong dan 43 duplikat.
- Preprocessing menurunkan token dari 59.635 menjadi 33.264 dan kosakata dari 8.997 menjadi 6.652.
- Noise analysis menandai 314 komentar (10,5%); 2.686 komentar (89,5%) bersih.
- Kata teratas: `ai` (2.230), `manusia` (781), `pakai` (262), `kerja` (224), `tahu` (174), `belajar` (165).
- Seluruh angka dan gambar berasal dari dataset hasil scraping nyata.

Laporan lengkap ada di `LAPORAN.md`.